In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')

In [ ]:
df.head(6)

In [ ]:
print(df.groupby('Sex')['Survived'].sum())

**more females survived**

In [ ]:
print(df.groupby('Sex')['Survived'].mean()*100)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df['AgeRange'] = pd.cut(df['Age'],bins =[0,12,19,45,60,100], labels=['child','teen','adult','old','very_old'])

In [ ]:
print(df.groupby('AgeRange')['Survived'].sum())
print(df.groupby('AgeRange')['Survived'].mean())

In [ ]:
print(df.isnull().sum())
print(df.isnull().mean()*100)

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()}")

In [ ]:
print(df.groupby('Parch')['Survived'].mean())

In [ ]:
#Extract title
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')

#survival rate per title
print(df.groupby('Title')['Survived'].mean().sort_values(ascending=False))

In [ ]:
df['Has_Cabin'] = df['Cabin'].notna().astype(int)

In [ ]:
df.drop(columns=['PassengerId', 'Name', 'Cabin', 'AgeRange'], inplace=True)

In [ ]:
df

In [ ]:
common_titles = ['Mr', 'Miss', 'Mrs', 'Master']
df['Title'] = df['Title'].apply(lambda x: x if x in common_titles else 'Other')

print(df['Title'].value_counts())

In [ ]:
df.drop(columns=['Ticket'], inplace=True)
print(df.columns.tolist())

In [ ]:
#fill with S, bcoz it appeared mosttime
df['Embarked'] = df['Embarked'].fillna('S')

#fill empty age with their title group median
df['Age'] = df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1  # +1 for the passenger themselves

# Check survival rate by family size
print(df.groupby('FamilySize')['Survived'].mean())

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1  # +1 for the passenger themselves

# survival rate by family size
print(df.groupby('FamilySize')['Survived'].mean())

In [ ]:
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Check
print(df['IsAlone'].value_counts())
print(df.groupby('IsAlone')['Survived'].mean())

In [ ]:
df.drop(columns=['SibSp', 'Parch'], inplace=True)
print(df.columns.tolist())

In [ ]:
df = df.astype({col: int for col in df.select_dtypes(bool).columns})
print(df.head())# Label encode Sex
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# One-hot encode 
df = pd.get_dummies(df, columns=['Embarked', 'Title'], drop_first=False)

# Verify
print(df.shape)

In [ ]:
df = df.astype({col: int for col in df.select_dtypes(bool).columns})
print(df.head())

**TRAIN MODEL**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Split
X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(classification_report(y_test, y_pred))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(feature_importance)

In [ ]:
import os

# Find all available input files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
test_df = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
passenger_ids = test_df['PassengerId']

test_df['Title'] = test_df['Name'].str.extract(r',\s*(\w+)\.')
test_df['Title'] = test_df['Title'].apply(lambda x: x if x in ['Mr','Miss','Mrs','Master'] else 'Other')
test_df['Has_Cabin'] = test_df['Cabin'].notna().astype(int)
test_df['FamilySize'] = test_df['SibSp'] + test_df['Parch'] + 1
test_df['IsAlone'] = (test_df['FamilySize'] == 1).astype(int)
test_df.drop(columns=['PassengerId','Name','Cabin','Ticket','SibSp','Parch'], inplace=True)
test_df['Embarked'] = test_df['Embarked'].fillna('S')
test_df['Age'] = test_df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))
test_df['Fare'] = test_df['Fare'].fillna(test_df['Fare'].median())
test_df['Sex'] = test_df['Sex'].map({'male': 0, 'female': 1})
test_df = pd.get_dummies(test_df, columns=['Embarked','Title'], drop_first=False)
test_df = test_df.astype({col: int for col in test_df.select_dtypes(bool).columns})
test_df = test_df.reindex(columns=X.columns.tolist(), fill_value=0)

pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': rf_model.predict(test_df)
}).to_csv('submission.csv', index=False)
